# Per-step hooks in a generation loop

To record a whole generation as one trace, pass the bound method: `tl.trace(model.generate, ...)` ([guide](../../docs/guides/kv_cache.md#trace-a-generation-loop)). This notebook does the other thing: it makes the decode loop explicit so each step runs under its own hook plan, on a tiny local generator with no download.

In [ ]:
# recipe-test: tool=torchlens expected=True
from __future__ import annotations

from collections.abc import Callable, Mapping

import torch
from torch import nn

import torchlens as tl

HookPlan = Mapping[object, object] | None


class TinyTokenGenerator(nn.Module):
    """Minimal token generator with a visible decode loop."""

    def __init__(self) -> None:
        super().__init__()
        self.embed = nn.Embedding(5, 4)
        self.transition = nn.Linear(4, 4)
        self.head = nn.Linear(4, 5)
        with torch.no_grad():
            self.embed.weight.zero_()
            self.transition.weight.zero_()
            self.transition.bias.zero_()
            self.head.weight.zero_()
            self.head.bias.zero_()
            self.head.bias[1] = 0.1  # with no other signal, token 1 wins
            self.head.weight[3, 0] = 1.0  # hidden feature 0 votes for token 3

    def forward(self, token: torch.Tensor) -> torch.Tensor:
        """Next-token logits ``(batch, vocab)`` from the current token ids ``(batch,)``."""
        hidden = torch.relu(self.transition(self.embed(token)))
        return self.head(hidden)

    def generate(
        self,
        start_token: torch.Tensor,
        *,
        steps: int,
        hook_factory: Callable[[int], HookPlan] | None = None,
    ) -> tuple[torch.Tensor, list[tl.Trace]]:
        """Decode greedily for ``steps`` steps, each under ``hook_factory(step)``; return the
        token sequence and one trace per step."""
        current = start_token.clone()
        tokens = [current]
        logs: list[tl.Trace] = []
        for step in range(steps):
            hooks = None if hook_factory is None else hook_factory(step)
            step_log = tl.trace(self, current, capture=tl.options.CaptureOptions(hooks=hooks))
            logits = step_log.output_ops[-1].out.detach()
            current = logits.argmax(dim=-1)
            tokens.append(current)
            logs.append(step_log)
        return torch.stack(tokens, dim=1), logs


model = TinyTokenGenerator().eval()
start = torch.tensor([0])
RESULT = model(start).shape == (1, 5)

The clean decode picks token `1` at every step because the output bias favors it. The steered decode leaves step 0 alone and, from step 1 on, adds a positive feature at the ReLU site, so token `3` wins from then. The same per-step choice can wrap a real model's `generate()`.

In [ ]:
# recipe-test: tool=torchlens expected=True
clean_tokens, clean_logs = model.generate(start, steps=3)
print("clean", clean_tokens)

steering_direction = torch.tensor([1.0, 0.0, 0.0, 0.0])


def hooks_for_step(step: int) -> HookPlan:
    """No hooks at step 0; steer the ReLU output from step 1 on."""
    if step == 0:
        return None
    return {tl.func("relu"): tl.steer(steering_direction, strength=1.0, dim=-1)}


steered_tokens, steered_logs = model.generate(start, steps=3, hook_factory=hooks_for_step)
print("steered", steered_tokens)
assert clean_tokens.tolist() == [[0, 1, 1, 1]]
assert steered_tokens.tolist() == [[0, 1, 3, 3]]
assert len(clean_logs) == len(steered_logs) == 3
assert not steered_logs[0].last_run_records()
assert steered_logs[1].last_run_records()[-1].edit_name == "steer"
RESULT = steered_tokens.tolist() == [[0, 1, 3, 3]]